In [9]:
import pyautogui as pyg
import time

pyg.FAILSAFE = True

print("2秒後開始執行，請切換至 Chrome 恐龍遊戲視窗...")
pyg.sleep(2)
pyg.press('space') # 自動按一下啟動遊戲

# 【優化 1】將截圖區域的 X 軸起點大幅拉遠到 420，寬度增加到 160
# 這樣就算到了 300、600 分速度極快，也有足夠的距離提前起跳
scan_region = (420, 580, 160, 110)

while True:
    im = pyg.screenshot(region=scan_region)
    
    # 穩定的日夜交替判斷（多點平均亮度）
    p1 = im.getpixel((10, 10))
    p2 = im.getpixel((80, 10))
    p3 = im.getpixel((150, 10))
    avg_brightness = (p1[0] + p2[0] + p3[0]) / 3
    is_night = (avg_brightness < 100) 
    
    action = None # 'jump' 或 'duck' 或 None
    
    # -------------------------------------------------------------
    # 1. 優先檢查【地面層】（仙人掌 / 低空障礙）：起跳動作
    # -------------------------------------------------------------
    for x in range(15, 140):
        for y in range(65, 95): # 加大地面掃描的垂直範圍，確保高矮仙人掌都能抓到
            c = im.getpixel((x, y))
            if is_night:
                if c[0] > 150 and c[1] > 150 and c[2] > 150:
                    action = 'jump'
                    break
            else:
                if c[0] < 100 and c[1] < 100 and c[2] < 100:
                    action = 'jump'
                    break
        if action:
            break
                
    # -------------------------------------------------------------
    # 2. 其次檢查【空中層】（中空鳥需要低頭，高空鳥保持不變）
    # -------------------------------------------------------------
    if not action:
        for x in range(15, 140):
            for y in range(15, 38): # 中空鳥高度（避開地面仙人掌頂端）
                c = im.getpixel((x, y))
                if is_night:
                    if c[0] > 150 and c[1] > 150 and c[2] > 150:
                        action = 'duck'
                        break
                else:
                    if c[0] < 100 and c[1] < 100 and c[2] < 100:
                        action = 'duck'
                        break
            if action:
                break
                    
    # -------------------------------------------------------------
    # 3. 執行動作與緩衝控制
    # -------------------------------------------------------------
    if action == 'jump':
        pyg.press('space')
        time.sleep(0.01) # 極短緩衝，應付高速連續障礙
    elif action == 'duck':
        pyg.keyDown('down')
        time.sleep(0.35)
        pyg.keyUp('down')

2秒後開始執行，請切換至 Chrome 恐龍遊戲視窗...


KeyboardInterrupt: 